# Build a regression model

The real estate agency from D01's lab wants a first price model. One row is one house sale in Ames, Iowa, 2006 to 2010. You fit a linear regression on one column, then on all of them, and report the errors in dollars.

## Step 1 · Run the starter

The cell applies D01 6.4's decisions: it reads the file with `keep_default_na=False, na_values=[""]` (so the word `None` stays a category), removes the 4 houses over 4,000 sq ft, takes `y = np.log(SalePrice)`, drops `Id`, `SaleType` and `SaleCondition`, and splits 75 / 25 with `random_state=42`. It also builds D01's three-block `ColumnTransformer` as `pre`, with `make_pre(X)` to build the same blocks for another set of columns (the bonus uses it). Check the number of training and test houses.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/ames.csv", keep_default_na=False, na_values=[""])
df = df[df["GrLivArea"] <= 4000]
y = np.log(df["SalePrice"])
X = df.drop(columns=["Id", "SalePrice", "SaleType", "SaleCondition"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# D01 6.4: "no garage" and the like are categories; other gaps are imputed inside the pipeline
STRUCTURAL = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
              "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC",
              "Fence", "MiscFeature"]


def make_pre(X):
    """D01 6.4's three-block ColumnTransformer for the columns X has."""
    num_cols = list(X.select_dtypes(include="number").columns)
    cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
    structural = [c for c in STRUCTURAL if c in cat_cols]
    other_cat = [c for c in cat_cols if c not in STRUCTURAL]
    blocks = [
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
        ("none", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"),
                               OneHotEncoder(handle_unknown="ignore", sparse_output=False)), structural),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore", sparse_output=False)), other_cat),
    ]
    return ColumnTransformer([b for b in blocks if b[2]])


pre = make_pre(X)
print(len(X_train), "training houses,", len(X_test), "test houses,", X.shape[1], "feature columns")

## Step 2 · "Roughly, what is a square foot worth here?"

Translate: fit `LinearRegression()` on `X_train[["GrLivArea"]]` against the price in dollars, `np.exp(y_train)`. Print `coef_` and `intercept_`, and predict the price of a 1,500 sq ft house.

## Step 3 · Score the one-column model in dollars

MAE, RMSE and R² on the test houses (`mean_absolute_error`, `root_mean_squared_error`, `r2_score`). Then score a `DummyRegressor()` (it always predicts the mean training price) the same way.

## Step 4 · Every column

`make_pipeline(pre, LinearRegression())`, fitted on `X_train, y_train` (the log price). Predict the test houses, turn the predictions back into dollars with `np.exp`, and print MAE, RMSE and R² in dollars. Print R² on the training houses too.

## Step 5 · The residuals

Plot predicted price on the x-axis and actual minus predicted on the y-axis. Find the house with the largest miss and print its living area, neighborhood, actual and predicted price.

## Step 6 · "How far off is a typical estimate?"

Answer in one sentence with the MAE from step 4, and add the median of `abs(predicted / actual - 1)` as a percentage.

## Bonus · `TotalSF`, then five columns

## Check yourself